[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soorig/research-blog-qc/blob/main/content/teaching/random-processes/Day2_Random_Processes.ipynb)

_위 배지를 클릭하면 Google Colab에서 바로 실행할 수 있습니다 (로컬 환경 설정 불필요)._

---

# Day 2 — 무작위(Random)와 동역학

오늘은 "무작위"가 시스템에 어떤 영향을 미치는지 격자 위 입자 모델을 통해 살펴봅니다.

**오늘의 핵심 질문**

1. 입자들이 무작위로 움직이면(Random Walk) 분포는 어떻게 변하는가?
2. 같은 무작위 시도라도 "이득이 되는 이동만 받아들이면(Greedy)" 결과는 어떻게 달라지는가?
3. 강화학습 에이전트가 위 두 단계를 **스스로 발견**할 수 있는가?
4. 두 입자를 동시에 교환하는 카와사키 동역학(Kawasaki Dynamics)에서는 어떤 차이가 나타나는가?

**오늘의 구성**

| 절 | 주제 | 공간 |
|---|---|---|
| 1-1 | Random Walk + Greedy | 사각 격자 |
| 1-2 | Random Walk + Greedy | 반(半) 환형 (슬라이스 토러스) |
| 1-3 | Random Walk + Greedy | 환형 (구멍 뚫린 원판) |
| 1-4 | DQN 강화학습 | 환형 |
| 2-1 | Kawasaki 동역학 | 환형 |
| 2-2 | Kawasaki 동역학 | 토러스 (3D) |

필요한 패키지를 설치합니다.

In [ ]:
!pip install numpy matplotlib IPython tqdm

# 1. Random Walk & Greedy on Different Spaces

## 핵심 개념

- **Random Walk (확산)**: 각 시점마다 입자는 이웃한 빈 칸 중 하나를 "무조건 무작위로" 선택하여 이동합니다. 결정 규칙이 없으므로 입자 분포는 시간이 지남에 따라 평형 분포에 수렴합니다.
- **Greedy (탐색·최적화)**: 동일한 무작위 시도를 하되, **이동 후 점수가 더 좋아질 때만** 받아들입니다. 점수 함수는 "이웃과 떨어져 있을수록 좋다" 같은 분리 척도(spacing)로 정의합니다.

> **정확한 "균일"의 의미** — 우리 모델은 *배제 과정(exclusion process)*입니다: 한 칸에는 입자가 0개 또는 1개만 있을 수 있습니다. 평형에서는 모든 "입자 배치"가 등확률로 나타나며, 그 결과 거시적 밀도장(coarse-grained density)은 균일해 보입니다. 단일 입자의 단순 랜덤워크라면 평형 분포는 격자의 차수(degree)에 비례하므로, 모서리·코너에서 약간 다른 값이 나오겠지만, 다입자 배제 과정에서는 이 효과가 평형에서 사라집니다.

두 단계를 "확산 → 그리디" 순으로 이어 붙이면, 처음에는 입자들이 공간 전체로 퍼졌다가, 이후에는 서로 거리를 두고 균등 배치되는 모습을 볼 수 있습니다. 물리학적으로 이것은 **고온(T = ∞) 완화** 후 **저온(T = 0) 완화**로 가는 시퀀스이며, §2의 카와사키와 자연스럽게 연결됩니다.

---

## 1-1. 사각 격자 (Square Grid)

### 1-1.1. 환경 정의

`RecLeftWallEnv`는 N×N 사각 격자 환경입니다.
- 셀 값 `1`: 빈 공간
- 셀 값 `-1`: 입자
- 셀 값 `2`: 벽 (이 환경에서는 사용되지 않음 — 모든 칸이 유효)

초기 상태에서는 모든 입자가 왼쪽 영역(`x < wall_limit`)에 모여 있습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from IPython.display import HTML
from tqdm import tqdm

class RecLeftWallEnv:
    def __init__(self, N=80, particle_ratio=0.25, wall_limit=16):
        self.N = N
        # 2: 벽/구멍 (이동불가), 1: 빈공간, -1: 입자
        self.lattice = np.ones((N, N)) * 1 # Initialize entire grid as empty space (1)

        # --- 1. 기하학적 구조 생성 (Square Grid) ---
        # 모든 격자 셀이 유효한 공간이 됩니다.
        self.valid_coords = [] # 입자가 존재할 수 있는 모든 좌표

        for x in range(N): # row
            for y in range(N): # col
                self.valid_coords.append((x, y))

        # --- 2. 입자 배치 (왼쪽 벽면 x < 16 에 집중 배치) ---
        # 전체 유효 면적 대비 입자 수 계산
        self.num_particles = int(len(self.valid_coords) * particle_ratio)

        count = 0
        # 왼쪽(열 인덱스 col < wall_limit)부터 훑으면서 입자 채우기
        # x좌표(가로)를 col로 봅니다.
        for col in range(wall_limit):
            for row in range(N):
                # 유효한 빈 공간(1)이라면 입자(-1) 배치
                if self.lattice[row, col] == 1:
                    if count < self.num_particles:
                        self.lattice[row, col] = -1
                        count += 1
                    else:
                        break # 입자를 다 채웠으면 중단
            if count >= self.num_particles:
                break

        print(f"초기화 완료: 총 {count}개의 입자가 x<{wall_limit} 영역에 배치됨.")

    def _get_neighbor_coords(self, x, y):
        """비주기적 경계 조건(Hard Wall) 이웃 좌표"""
        neighbors = [
            (x+1, y), (x-1, y),
            (x, y+1), (x, y-1)
        ]
        # 격자 범위 내에 있는 이웃만 반환
        valid_neighbors = []
        for nx, ny in neighbors:
            if 0 <= nx < self.N and 0 <= ny < self.N:
                valid_neighbors.append((nx, ny))
        return valid_neighbors

    def step_diffusion(self):
        """Phase 1: 단순 확산 (Random Walk)"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        # 이동 시도
        possible_moves = self._get_neighbor_coords(x, y)
        if not possible_moves: return # 이동 가능한 곳이 없으면 패스

        nx, ny = possible_moves[np.random.randint(len(possible_moves))]

        # 빈 공간(1)으로만 이동
        if self.lattice[nx, ny] == 1:
            self.lattice[x, y], self.lattice[nx, ny] = self.lattice[nx, ny], self.lattice[x, y]

    def step_greedy(self):
        """Phase 2: 고립 학습 (Anti-Clustering)"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        neighbors = self._get_neighbor_coords(x, y)
        if not neighbors: return # 이동 가능한 곳이 없으면 패스

        current_cost = sum([1 for nx, ny in neighbors if self.lattice[nx, ny] == -1])

        # 2. 이동 후보
        target_nx, target_ny = neighbors[np.random.randint(len(neighbors))]
        if self.lattice[target_nx, target_ny] != 1: return

        # 3. 미래 Cost 예측 (Virtual Move)
        # Swap
        self.lattice[x, y], self.lattice[target_nx, target_ny] = \
            self.lattice[target_nx, target_ny], self.lattice[x, y]

        new_neighbors = self._get_neighbor_coords(target_nx, target_ny)
        future_cost = sum([1 for nnx, nny in new_neighbors if self.lattice[nnx, nny] == -1])

        # 4. 결정
        if future_cost > current_cost:
            # Revert (Cost 증가 시 취소)
            self.lattice[x, y], self.lattice[target_nx, target_ny] = \
                self.lattice[target_nx, target_ny], self.lattice[x, y]


### 1-1.2. 시뮬레이션 — 확산 후 그리디

두 단계 시나리오:
1. **Phase 1 (Diffusion)**: 약 500 프레임 동안 무작위 이동 → 입자들이 격자 전체에 흔어집니다.
2. **Phase 2 (Greedy)**: 이후 "점수가 향상될 때만" 이동 → 입자들이 서로 거리를 두며 정렬됩니다.

각 프레임은 `STEPS_PER_FRAME = N²/2` 번의 시도를 압축합니다.

In [ ]:
# -------------------------- 시뮬레이션 설정 -------------------------------
N = 60

Total_SEPARATING = 1000
FRAMES_DIFFUSION = 500
FRAMES_Greedy = Total_SEPARATING - FRAMES_DIFFUSION
TOTAL_FRAMES = FRAMES_DIFFUSION + FRAMES_Greedy
STEPS_PER_FRAME = N * N // 2
# --------------------------------------------------------------------------
# 환경 초기화 (x < 16)
env = RecLeftWallEnv(N=N, particle_ratio=0.20, wall_limit=16)

# 컬러맵: -1(파랑:입자), 1(흰색:빈공간), 2(검정:벽)
cmap = colors.ListedColormap(['blue', 'white', 'black'])
bounds = [-1.5, -0.5, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(env.lattice, cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Init: Left Cluster (x < 16)")

print("영상 렌더링 시작...")

def animate(frame):
    current_phase = ""

    if frame < FRAMES_DIFFUSION:
        current_phase = "Phase 1: Diffusion (Filling the Ring)"
        for _ in range(STEPS_PER_FRAME):
            env.step_diffusion()
    else:
        current_phase = "Phase 2: Greedy Optimization (Spacing)"
        for _ in range(STEPS_PER_FRAME):
            env.step_greedy()

    im.set_data(env.lattice)
    title.set_text(f"{current_phase} | Frame: {frame}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=TOTAL_FRAMES, interval=40, blit=True)
plt.close()
HTML(anim.to_html5_video())

## 1-2. 슬라이스 토러스 (반환형, Half-Annulus)

### 1-2.1. 환경 정의

공간이 "반환형" 모양일 때는 어떨까요? 입자는 도넛의 절반만 사용할 수 있고, 오른쪽 절반은 벽으로 막혀 있습니다.

- 형태: 환형(annulus)의 왼쪽 절반만 유효 영역
- 외경: `(N//2) - 2`, 내경: 외경의 1/3

이렇게 되면 입자가 단순히 "오른쪽으로 퍼진다"가 아니라 곡선 경로를 따라 흐릅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from IPython.display import HTML
from tqdm import tqdm

class AnnulusHalfLeftEnv:
    def __init__(self, N=80, particle_ratio=0.25, wall_limit=16):
        self.N = N
        # 2: 벽/구멍 (이동불가), 1: 빈공간, -1: 입자
        self.lattice = np.ones((N, N)) * 2

        # --- 1. 기하학적 구조 생성 (Half Annulus) ---
        cx, cy = N // 2, N // 2
        self.R_out = (N // 2) - 2
        self.R_in = self.R_out / 3.0

        self.valid_coords = []

        for x in range(N): # row
            for y in range(N): # col
                dist = np.sqrt((x - cx)**2 + (y - cy)**2)

                # [수정됨] 링 내부에 있고 AND 왼쪽 절반(y < cy)에만 빈 공간 생성
                is_in_ring = self.R_in <= dist <= self.R_out
                is_left_half = y < cy  # 오른쪽 절반으로 넘어가지 못하게 하는 핵심 조건

                if is_in_ring and is_left_half:
                    self.lattice[x, y] = 1 # 빈 공간
                    self.valid_coords.append((x, y))
                else:
                    self.lattice[x, y] = 2 # 벽 (오른쪽 절반 포함)

        # --- 2. 입자 배치 (왼쪽 벽면 x < 16 에 집중 배치) ---
        self.num_particles = int(len(self.valid_coords) * particle_ratio)

        count = 0
        for col in range(wall_limit):
            for row in range(N):
                if self.lattice[row, col] == 1:
                    if count < self.num_particles:
                        self.lattice[row, col] = -1
                        count += 1
                    else:
                        break
            if count >= self.num_particles:
                break

        print(f"초기화 완료: 총 {count}개의 입자가 배치됨 (오른쪽 진입 불가).")

    def _get_neighbor_coords(self, x, y):
        """이웃 좌표 반환"""
        # 오른쪽 절반이 벽(2)으로 막혀있으므로,
        # PBC(Periodic Boundary Condition)를 유지해도 넘어가다가 벽에 막혀서 못 갑니다.
        return [
            ((x+1)%self.N, y), ((x-1)%self.N, y),
            (x, (y+1)%self.N), (x, (y-1)%self.N)
        ]

    def step_diffusion(self):
        """Phase 1: 단순 확산"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        # 랜덤 이동 시도
        nx, ny = self._get_neighbor_coords(x, y)[np.random.randint(4)]

        # 빈 공간(1)인 경우에만 이동 (오른쪽 절반은 2라서 이동 불가)
        if self.lattice[nx, ny] == 1:
            self.lattice[x, y], self.lattice[nx, ny] = self.lattice[nx, ny], self.lattice[x, y]

    def step_greedy(self):
        """Phase 2: 고립 학습"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        neighbors = self._get_neighbor_coords(x, y)
        current_cost = sum([1 for nx, ny in neighbors if self.lattice[nx, ny] == -1])

        target_nx, target_ny = neighbors[np.random.randint(4)]

        # 빈 공간(1)이 아니면(즉, 벽이거나 다른 입자면) 리턴
        if self.lattice[target_nx, target_ny] != 1: return

        # Swap & Check Logic
        self.lattice[x, y], self.lattice[target_nx, target_ny] = \
            self.lattice[target_nx, target_ny], self.lattice[x, y]

        new_neighbors = self._get_neighbor_coords(target_nx, target_ny)
        future_cost = sum([1 for nnx, nny in new_neighbors if self.lattice[nnx, nny] == -1])

        if future_cost > current_cost:
            # Revert
            self.lattice[x, y], self.lattice[target_nx, target_ny] = \
                self.lattice[target_nx, target_ny], self.lattice[x, y]



### 1-2.2. 시뮬레이션

확산 단계만 실행해 입자들이 반환형 공간을 어떻게 채우는지 관찰합니다.

In [ ]:
# ----------------------- 시뮬레이션 실행 설정 ----------------------
N = 60
Total_SEPARATING = 800
FRAMES_DIFFUSION = 800
FRAMES_Greedy = Total_SEPARATING - FRAMES_DIFFUSION
TOTAL_FRAMES = FRAMES_DIFFUSION + FRAMES_Greedy
STEPS_PER_FRAME = N * N // 2
#--------------------------------------------------------------------

# 환경 생성 (Half Left)
env = AnnulusHalfLeftEnv(N=N, particle_ratio=0.20, wall_limit=16)

cmap = colors.ListedColormap(['blue', 'white', 'black'])
bounds = [-1.5, -0.5, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(env.lattice, cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Half Annulus (Right Side Blocked)")

print("영상 렌더링 시작...")

def animate(frame):
    current_phase = ""

    if frame < FRAMES_DIFFUSION:
        current_phase = "Phase 1: Diffusion"
        for _ in range(STEPS_PER_FRAME):
            env.step_diffusion()
    else:
        current_phase = "Phase 2: Greedy Optimization"
        for _ in range(STEPS_PER_FRAME):
            env.step_greedy()

    im.set_data(env.lattice)
    title.set_text(f"{current_phase} | Frame: {frame}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=TOTAL_FRAMES, interval=40, blit=True)
plt.close()
HTML(anim.to_html5_video())

## 1-3. 구멍 뚫린 원판 (Punctured Disk / Annulus)

### 1-3.1. 환경 정의

이번에는 도넛 전체를 사용합니다. 가운데 작은 원이 비어 있는 환형 공간입니다.

이 공간은 단순 연결되지 않은(non-simply-connected) 위상 — $\pi_1 \cong \mathbb{Z}$ — 을 가집니다. 즉 "구멍 주위를 한 바퀴 도는 경로"가 점으로 수축되지 않습니다.

> **위상이 어디서 보이는가** — 위상의 비자명성은 *입자 궤적의 통계*에서 가장 분명히 나타납니다. 예를 들어, 단일 랜덤워커가 시간 $t$ 까지 구멍을 "몇 바퀴" 감았는지 (winding number)는 평균 $0$, 분산 $\sim t$ 을 갖는 무작위 변수입니다 — 이 효과는 단순 연결 공간에는 없습니다. 반면 *평형 밀도장* (coarse-grained density profile)은 위상에 무관하며, 균일 배제 과정의 평형은 단순한 환형이든 단순 연결 영역이든 "가용 셀 위의 균일 분포"입니다. 본 시뮬레이션의 시각화는 후자(밀도)를 보여주므로, "구멍 효과"를 시각적으로 직접 확인하기는 어렵습니다 — 비교를 위해 1-1, 1-2와 함께 보면 형태(geometry)에 따른 *과도 거동(transient behavior)*의 차이를 느낄 수 있습니다.

두 클래스를 정의합니다.
- `AnnulusSimulation`: 일반화된 시뮬레이션 클래스
- `AnnulusLeftWallEnv`: 왼쪽 영역에 입자 클러스터가 미리 모인 초기조건

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from tqdm import tqdm

class AnnulusSimulation:

    def __init__(self, N=100, particle_ratio=0.20, wall_limit=16):
        """
        Args:
            N (int): 격자 크기 (NxN)
            particle_ratio (float): 유효 공간 대비 입자 비율 (0.0 ~ 1.0)
            wall_limit (int): 초기화 시 입자를 배치할 왼쪽 벽면의 x좌표 한계
        """
        self.N = N
        self.particle_ratio = particle_ratio
        self.wall_limit = wall_limit

        # 격자 상태 정의 (2:벽, 1:빈공간, -1:입자)
        self.lattice = np.ones((N, N)) * 2
        self.valid_coords = [] # 입자가 이동 가능한 좌표 리스트

        self._init_geometry()
        self._init_particles()

    def _init_geometry(self):
        """기하학적 구조 생성 (중앙이 1/9만큼 빈 도넛 모양)"""
        cx, cy = self.N // 2, self.N // 2
        self.R_out = (self.N // 2) - 2
        self.R_in = self.R_out / 3.0 # 면적 1/9 조건을 위해 반지름 1/3 설정

        for x in range(self.N):
            for y in range(self.N):
                # 중심으로부터의 유클리드 거리
                dist = np.sqrt((x - cx)**2 + (y - cy)**2)

                # 도넛 내부 영역만 빈 공간(1)으로 설정
                if self.R_in <= dist <= self.R_out:
                    self.lattice[x, y] = 1
                    self.valid_coords.append((x, y))
                else:
                    self.lattice[x, y] = 2

    def _init_particles(self):
        """입자 초기화: 왼쪽 벽면(x < wall_limit)에 밀집 배치"""
        num_valid = len(self.valid_coords)
        self.num_particles = int(num_valid * self.particle_ratio)

        count = 0
        # x좌표(col)를 0부터 wall_limit까지 훑으며 채움
        for col in range(self.wall_limit):
            for row in range(self.N):
                if self.lattice[row, col] == 1:
                    if count < self.num_particles:
                        self.lattice[row, col] = -1
                        count += 1
                    else:
                        return # 배치 완료 시 종료

        print(f"[초기화 완료] 격자: {self.N}x{self.N} | 입자 수: {count}개 | 초기 위치: x < {self.wall_limit}")

    def _get_neighbors(self, x, y):
        """주기적 경계 조건(PBC)을 적용한 4방향 이웃 좌표"""
        return [
            ((x+1)%self.N, y), ((x-1)%self.N, y),
            (x, (y+1)%self.N), (x, (y-1)%self.N)
        ]

    def step_diffusion(self):
        """1회의 확산 스텝 (Random Walk)"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return # 입자가 아니면 패스

        nx, ny = self._get_neighbors(x, y)[np.random.randint(4)]

        # 빈 공간(1)으로만 이동 가능
        if self.lattice[nx, ny] == 1:
            self.lattice[x, y], self.lattice[nx, ny] = self.lattice[nx, ny], self.lattice[x, y]

    def step_greedy(self):
        """1회의 greedy 스텝 (Anti-Clustering / Isolation)"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return # 입자가 아니면 패스

        neighbors = self._get_neighbors(x, y)
        current_cost = sum([1 for nx, ny in neighbors if self.lattice[nx, ny] == -1])

        # 이동할 타겟 랜덤 선택
        target_nx, target_ny = neighbors[np.random.randint(4)]
        if self.lattice[target_nx, target_ny] != 1: return # 빈 공간이 아니면 패스

        # 가상 이동 (Swap)
        self.lattice[x, y], self.lattice[target_nx, target_ny] = \
            self.lattice[target_nx, target_ny], self.lattice[x, y]

        # 미래 Cost 계산
        new_neighbors = self._get_neighbors(target_nx, target_ny)
        future_cost = sum([1 for nnx, nny in new_neighbors if self.lattice[nnx, nny] == -1])

        # Cost가 증가하면 원상복구 (Reject)
        if future_cost > current_cost:
            self.lattice[x, y], self.lattice[target_nx, target_ny] = \
                self.lattice[target_nx, target_ny], self.lattice[x, y]

    def run_simulation(self, diffusion_frames=200, greedy_frames=150):
        """시뮬레이션 전체 실행 (Diffusion -> Greedy)"""
        steps_per_frame = (self.N * self.N) // 2

        # Phase 1: Diffusion
        print(f"Phase 1: 확산 진행 중... ({diffusion_frames} frames)")
        for _ in tqdm(range(diffusion_frames * steps_per_frame)):
            self.step_diffusion()

        # Phase 2: Greedy Optimization
        print(f"Phase 2: 고립 학습 진행 중... ({greedy_frames} frames)")
        for _ in tqdm(range(greedy_frames * steps_per_frame)):
            self.step_greedy()

        print("시뮬레이션 완료.")

    def plot_state(self, title_suffix=""):
        """현재 상태 시각화"""
        cmap = colors.ListedColormap(['blue', 'white', 'black'])
        bounds = [-1.5, -0.5, 1.5, 2.5]
        norm = colors.BoundaryNorm(bounds, cmap.N)

        plt.figure(figsize=(8, 8))
        plt.imshow(self.lattice, cmap=cmap, norm=norm)
        plt.title(f"Simulation State {title_suffix}\n(Blue: Particles, White: Empty, Black: Wall)", fontsize=14)
        plt.axis('off')
        plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from IPython.display import HTML
from tqdm import tqdm

class AnnulusLeftWallEnv:
    def __init__(self, N=80, particle_ratio=0.25, wall_limit=16):
        self.N = N
        # 2: 벽/구멍 (이동불가), 1: 빈공간, -1: 입자
        self.lattice = np.ones((N, N)) * 2

        # --- 1. 기하학적 구조 생성 (Annulus with 1/9 Hole) ---
        cx, cy = N // 2, N // 2
        self.R_out = (N // 2) - 2
        self.R_in = self.R_out / 3.0 # 면적 1/9

        self.valid_coords = [] # 입자가 존재할 수 있는 모든 좌표 (링 내부)

        for x in range(N): # row
            for y in range(N): # col
                # 중심으로부터의 거리 계산
                # (x가 행, y가 열이라고 가정할 때 기하학적 x좌표는 y입니다)
                dist = np.sqrt((x - cx)**2 + (y - cy)**2)

                if self.R_in <= dist <= self.R_out:
                    self.lattice[x, y] = 1 # 빈 공간으로 초기화
                    self.valid_coords.append((x, y))
                else:
                    self.lattice[x, y] = 2 # 벽

        # --- 2. 입자 배치 (왼쪽 벽면 x < 16 에 집중 배치) ---
        # 전체 유효 면적 대비 입자 수 계산
        self.num_particles = int(len(self.valid_coords) * particle_ratio)

        count = 0
        # 왼쪽(열 인덱스 col < wall_limit)부터 훑으면서 입자 채우기
        # x좌표(가로)를 col로 봅니다.
        for col in range(wall_limit):
            for row in range(N):
                # 유효한 빈 공간(1)이라면 입자(-1) 배치
                if self.lattice[row, col] == 1:
                    if count < self.num_particles:
                        self.lattice[row, col] = -1
                        count += 1
                    else:
                        break # 입자를 다 채웠으면 중단
            if count >= self.num_particles:
                break

        print(f"초기화 완료: 총 {count}개의 입자가 x<{wall_limit} 영역에 배치됨.")

    def _get_neighbor_coords(self, x, y):
        """토러스 위상(PBC) 이웃 좌표"""
        return [
            ((x+1)%self.N, y), ((x-1)%self.N, y),
            (x, (y+1)%self.N), (x, (y-1)%self.N)
        ]

    def step_diffusion(self):
        """Phase 1: 단순 확산 (Random Walk)"""
        # 전체 링 좌표 중 하나 랜덤 선택 (효율을 위해 valid_coords에서 샘플링 후 확인)
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        # 이동 시도
        nx, ny = self._get_neighbor_coords(x, y)[np.random.randint(4)]

        # 빈 공간(1)으로만 이동
        if self.lattice[nx, ny] == 1:
            self.lattice[x, y], self.lattice[nx, ny] = self.lattice[nx, ny], self.lattice[x, y]

    def step_greedy(self):
        """Phase 2: 고립 학습 (Anti-Clustering)"""
        idx = np.random.choice(len(self.valid_coords))
        x, y = self.valid_coords[idx]

        if self.lattice[x, y] != -1: return

        # 1. 현재 Cost
        neighbors = self._get_neighbor_coords(x, y)
        current_cost = sum([1 for nx, ny in neighbors if self.lattice[nx, ny] == -1])

        # 2. 이동 후보
        target_nx, target_ny = neighbors[np.random.randint(4)]
        if self.lattice[target_nx, target_ny] != 1: return

        # 3. 미래 Cost 예측 (Virtual Move)
        # Swap
        self.lattice[x, y], self.lattice[target_nx, target_ny] = \
            self.lattice[target_nx, target_ny], self.lattice[x, y]

        new_neighbors = self._get_neighbor_coords(target_nx, target_ny)
        future_cost = sum([1 for nnx, nny in new_neighbors if self.lattice[nnx, nny] == -1])

        # 4. 결정
        if future_cost > current_cost:
            # Revert (Cost 증가 시 취소)
            self.lattice[x, y], self.lattice[target_nx, target_ny] = \
                self.lattice[target_nx, target_ny], self.lattice[x, y]



### 1-3.2. 시뮬레이션 (Ring)

Diffusion은 1프레임으로 두고 Greedy를 길게 돌립니다. 이미 입자가 모여 있는 상태에서 "분리" 과정만 보고 싶기 때문입니다.

In [ ]:
# --- 시뮬레이션 설정 ---
N = 60
# 입자가 한쪽에 몰려있으므로 확산 시간을 충분히 줍니다.
Total_SEPARATING = 1000
FRAMES_DIFFUSION = 1
FRAMES_Greedy = Total_SEPARATING - FRAMES_DIFFUSION
TOTAL_FRAMES = FRAMES_DIFFUSION + FRAMES_Greedy
STEPS_PER_FRAME = N * N // 2

# 환경 초기화 (x < 16)
env = AnnulusLeftWallEnv(N=N, particle_ratio=0.20, wall_limit=16)

# 컬러맵: -1(파랑:입자), 1(흰색:빈공간), 2(검정:벽)
cmap = colors.ListedColormap(['blue', 'white', 'black'])
bounds = [-1.5, -0.5, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(env.lattice, cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Init: Left Cluster (x < 16)")

print("영상 렌더링 시작...")

def animate(frame):
    current_phase = ""

    if frame < FRAMES_DIFFUSION:
        current_phase = "Phase 1: Diffusion (Filling the Ring)"
        for _ in range(STEPS_PER_FRAME):
            env.step_diffusion()
    else:
        current_phase = "Phase 2: Greedy Optimization (Spacing)"
        for _ in range(STEPS_PER_FRAME):
            env.step_greedy()

    im.set_data(env.lattice)
    title.set_text(f"{current_phase} | Frame: {frame}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=TOTAL_FRAMES, interval=40, blit=True)
plt.close()
HTML(anim.to_html5_video())

## 1-4. 강화학습으로 같은 일을 시켜보기 (DQN on Annulus)

### 1-4.1. 강화학습 시뮬레이션

지금까지는 사람이 "확산해라", "점수가 오를 때만 움직여라" 라고 규칙을 명시했습니다. 그런데 같은 결과를 **에이전트가 스스로** 학습할 수 있을까요?

**문제 설정**
- 상태: 입자 주변의 작은 패치(view) — 자기 위치를 중심으로 한 지역 정보
- 행동: 5가지 — 상/하/좌/우/제자리
- 보상: 이웃과의 분리도가 좋아지면 양(+), 나빠지면 음(−)
- 알고리즘: **DQN (Deep Q-Network)** + 경험 재생 + ε-greedy 탐색

유틸리티 `get_action_random_tiebreaker` 는 Q값이 동률일 때 무작위로 한 행동을 고릅니다. 이 작은 무작위성이 학습 초기 탐색 다양성을 높여 줍니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors, animation
from IPython.display import HTML
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from collections import deque
from tqdm import tqdm

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 0. 유틸리티 함수 (Tie-breaker)
# ==========================================
def get_action_random_tiebreaker(q_values):
    """
    Q값들 중 최대값을 가진 인덱스를 찾되,
    최대값이 여러 개라면 그 중 무작위로 하나를 선택함.
    q_values: tensor([[q0, q1, q2, q3, q4]]) (Batch size 1)
    """
    q_tensor = q_values.squeeze() # (1, 5) -> (5,)

    # 가장 큰 Q값 찾기
    max_q_value = torch.max(q_tensor)

    # 가장 큰 Q값과 같은 값을 가진 모든 인덱스 찾기
    # 예: [10, 5, 10, 2, 10] -> indices: [0, 2, 4]
    max_indices = torch.nonzero(q_tensor == max_q_value, as_tuple=True)[0]

    # 후보 인덱스들 중 하나 무작위 선택
    if len(max_indices) > 1:
        random_index = max_indices[torch.randint(len(max_indices), (1,))].item()
    else:
        random_index = max_indices.item()

    return random_index

# ==========================================
# 1. DQN 모델 정의 (CNN)
# ==========================================
class DQN(nn.Module):
    def __init__(self, output_dim):
        super(DQN, self).__init__()
        # 입력: 2채널 (채널0: 입자 위치, 채널1: 벽 위치), 7x7 크기
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, output_dim) # Action: 5 (상하좌우+정지)

    def forward(self, x):
        # x shape: (Batch, 2, 7, 7)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1) # Flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# ==========================================
# 2. 강화학습 환경 (Annulus Environment)
# ==========================================
class AnnulusEnv:
    def __init__(self, N=60, particle_ratio=0.2):
        self.N = N
        self.action_space = 5 # 0:Up, 1:Down, 2:Left, 3:Right, 4:Stay
        self.moves = [(-1, 0), (1, 0), (0, -1), (0, 1), (0, 0)]

        # 기하학적 구조 생성
        self.grid = np.zeros((N, N)) # 0: 빈공간
        self.particles = [] # 입자 좌표 리스트

        cx, cy = N // 2, N // 2
        R_out = (N // 2) - 2
        R_in = R_out / 3.0

        self.valid_coords = []
        for r in range(N):
            for c in range(N):
                dist = np.sqrt((r - cx)**2 + (c - cy)**2)
                if R_in <= dist <= R_out:
                    self.valid_coords.append((r, c))
                else:
                    self.grid[r, c] = 2 # 2: 벽(Wall)

        # 입자 초기화 (왼쪽 벽면에 몰아서 배치 - 학습 효과 극대화 확인용)
        num_particles = int(len(self.valid_coords) * particle_ratio)
        count = 0
        wall_limit = 20

        # x좌표(col) 기준 정렬된 valid coords 사용
        sorted_valid = sorted(self.valid_coords, key=lambda p: p[1])

        for r, c in sorted_valid:
            if c < wall_limit and count < num_particles:
                self.grid[r, c] = 1 # 1: 입자(Particle)
                self.particles.append([r, c])
                count += 1

        self.num_particles = len(self.particles)

    def get_observation(self, r, c, window_size=7):
        """특정 입자(r,c) 주변의 7x7 로컬 정보를 2채널 텐서로 반환"""
        pad = window_size // 2

        # 패딩된 그리드 생성 (Boundary 처리 단순화)
        padded_grid = np.pad(self.grid, pad, mode='constant', constant_values=2) # 외곽은 벽으로 처리

        # 로컬 윈도우 추출
        # 원래 좌표 (r, c)는 패딩된 그리드에서 (r+pad, c+pad)
        slice_window = padded_grid[r:r+window_size, c:c+window_size]

        # 채널 분리 (CNN 입력용)
        # Channel 0: 입자 (내 자신 제외한 다른 입자들) -> 1
        # Channel 1: 벽 -> 1
        obs = np.zeros((2, window_size, window_size), dtype=np.float32)

        # 채널 0: 입자 위치 (값이 1인 곳)
        obs[0] = (slice_window == 1).astype(float)
        # 내 자신(중앙)은 관측에서 '다른 입자'로 보이면 안되므로 0으로 지움
        obs[0, pad, pad] = 0

        # 채널 1: 벽 위치 (값이 2인 곳)
        obs[1] = (slice_window == 2).astype(float)

        return torch.FloatTensor(obs).unsqueeze(0).to(device) # (1, 2, 7, 7)

    def step(self, particle_idx, action):
        """특정 입자에 대해 행동 수행"""
        r, c = self.particles[particle_idx]

        # 1. 현재 상태의 이웃 수 계산 (보상 비교용)
        current_neighbors = self._count_neighbors(r, c)

        # 2. 이동할 위치 계산
        dr, dc = self.moves[action]
        nr, nc = r + dr, c + dc

        # 3. 충돌 체크 및 이동
        reward = 0
        moved = False

        # 범위 밖이나 벽(2)이거나 다른 입자(1)가 있으면 이동 불가
        if not (0 <= nr < self.N and 0 <= nc < self.N):
            reward = -5 # 격자 밖 페널티
        elif self.grid[nr, nc] == 2:
            reward = -5 # 벽 충돌 페널티
        elif self.grid[nr, nc] == 1 and (nr != r or nc != c):
            reward = -5 # 다른 입자 충돌 페널티
        else:
            # 이동 가능 (빈 공간이거나 제자리)
            if action != 4: # Stay가 아니면 실제 이동 처리
                self.grid[r, c] = 0   # 기존 위치 비움
                self.grid[nr, nc] = 1 # 새 위치 채움
                self.particles[particle_idx] = [nr, nc]
                moved = True

            # 4. 이동 후(또는 Stay 후) 이웃 수 계산
            new_neighbors = self._count_neighbors(nr, nc)

            # 5. Reward Function (핵심: 이웃 감소 = 보상)
            # 이웃 하나당 -1점 (Cost). 즉, 이웃이 줄면 점수 상승
            # 예: 이웃 3 -> 1 (Reward: -1 - (-3) = +2)
            # 추가: 이웃이 0개면 보너스 +1

            diff = current_neighbors - new_neighbors
            reward = diff * 2.0  # 개선된 만큼 보상

            if new_neighbors == 0:
                reward += 2.0 # 고립 성공 보너스
            else:
                reward -= 0.5 # 아직 이웃이 있으면 약간의 페널티 (빨리 떨어져라)

            # Stay(4)를 했는데 이웃이 여전히 많으면 페널티 (게으름 방지)
            if action == 4 and new_neighbors > 0:
                reward -= 1.0

        return reward, moved

    def _count_neighbors(self, r, c):
        cnt = 0
        for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < self.N and 0 <= nc < self.N:
                if self.grid[nr, nc] == 1:
                    cnt += 1
        return cnt

# ==========================================
# 3. 학습 루프 (Training Loop)
# ==========================================

# 하이퍼파라미터
N = 60
EPISODES = 500
STEPS_PER_EPISODE = 500  # 한 에피소드당 입자들을 몇 번 움직일지
BATCH_SIZE = 64
GAMMA = 0.9
EPSILON = 1.0
EPSILON_DECAY = 0.995
MIN_EPSILON = 0.05
LR = 0.001
MEMORY_CAPACITY = 10000

env = AnnulusEnv(N=N, particle_ratio=0.2)
policy_net = DQN(env.action_space).to(device)
target_net = DQN(env.action_space).to(device)
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=LR)
memory = deque(maxlen=MEMORY_CAPACITY)

loss_history = []
reward_history = []

print("강화학습 시작... (DQN Training)")

for episode in tqdm(range(EPISODES)):
    total_reward = 0

    # 에피소드마다 환경을 약간씩 리셋하거나 그냥 계속 이어서 학습(Continuous)
    # 여기서는 입자 위치를 유지하며 계속 학습 (Online Learning 형태)

    for _ in range(STEPS_PER_EPISODE):
        # 1. 랜덤한 입자 하나 선택 (Asynchronous Update)
        p_idx = np.random.randint(env.num_particles)
        r, c = env.particles[p_idx]

        # 2. 상태 관측
        state = env.get_observation(r, c)

        # 3. 행동 선택 (Epsilon Greedy)
        if random.random() < EPSILON:
            action = random.randint(0, 4)
        else:
            with torch.no_grad():
                q_values = policy_net(state)
                # [수정됨] argmax 대신 tie-breaker 함수 사용
                action = get_action_random_tiebreaker(q_values)

        # 4. 행동 수행
        reward, moved = env.step(p_idx, action)
        total_reward += reward

        # 5. 다음 상태 관측
        nr, nc = env.particles[p_idx]
        next_state = env.get_observation(nr, nc)

        # 6. 메모리 저장
        memory.append((state, action, reward, next_state))

        # 7. 학습 (Replay Buffer)
        if len(memory) > BATCH_SIZE:
            batch = random.sample(memory, BATCH_SIZE)
            b_state = torch.cat([b[0] for b in batch])
            b_action = torch.tensor([b[1] for b in batch], device=device).unsqueeze(1)
            b_reward = torch.tensor([b[2] for b in batch], device=device).unsqueeze(1)
            b_next_state = torch.cat([b[3] for b in batch])

            # Q(s, a)
            q_val = policy_net(b_state).gather(1, b_action)

            # Target Q: r + gamma * max(Q(s', a'))
            with torch.no_grad():
                max_next_q = target_net(b_next_state).max(1)[0].unsqueeze(1)
                target_q = b_reward + GAMMA * max_next_q

            loss = F.mse_loss(q_val, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Target Net 업데이트
    if episode % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    # Epsilon 감소
    EPSILON = max(MIN_EPSILON, EPSILON * EPSILON_DECAY)
    reward_history.append(total_reward)

print("학습 완료!")

# 학습 곡선 확인
plt.plot(reward_history)
plt.title("Training Reward Curve")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.show()

# ==========================================
# 4. 학습된 모델로 시뮬레이션 (Inference)
# ==========================================

# 테스트용 환경 새로 생성 (초기화)
test_env = AnnulusEnv(N=N, particle_ratio=0.2)
frames = []
test_steps = 400
steps_per_frame = N * 2 # 프레임당 움직임 횟수

print("학습된 에이전트로 시뮬레이션 생성 중...")

frames.append(test_env.grid.copy()) # 초기 상태

for t in tqdm(range(test_steps)):
    # 프레임당 여러 번 움직여서 속도감 있게
    for _ in range(steps_per_frame):
        p_idx = np.random.randint(test_env.num_particles)
        r, c = test_env.particles[p_idx]
        state = test_env.get_observation(r, c)

        # 학습된 모델로 행동 결정 (Greedy)
        with torch.no_grad():
            q_values = policy_net(state)
            # [수정됨] argmax 대신 tie-breaker 함수 사용
            action = get_action_random_tiebreaker(q_values)

        test_env.step(p_idx, action)

    frames.append(test_env.grid.copy())

# ==========================================
# 5. 애니메이션 생성
# ==========================================
cmap = colors.ListedColormap(['white', 'blue', 'black']) # 0:빈공간, 1:입자, 2:벽
bounds = [-0.5, 0.5, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(frames[0], cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Initial State")

def animate(idx):
    im.set_data(frames[idx])
    title.set_text(f"Trained Agent Simulation | Frame: {idx}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())

### 1-4.2. 학습된 모델로 시뮬레이션 (Inference)

학습이 끝난 `policy_net` 으로 ε-greedy 없이 **순수 그리디** 추론을 돌립니다. 학습 시 손으로 짜둔 그리디 규칙과 비슷한 분리 패턴이 나오는지 확인합니다.

In [ ]:

# ==========================================
# 4. 학습된 모델로 시뮬레이션 (Inference)
# ==========================================

#------------------------시뮬레이션 설정-------------------------------
test_env = AnnulusEnv(N=N, particle_ratio=0.2)
frames = []
test_steps = 500
steps_per_frame = N * 2 # 프레임당 움직임 횟수
#-----------------------------------------------------------------------
print("학습된 에이전트로 시뮬레이션 생성 중...")

frames.append(test_env.grid.copy()) # 초기 상태

for t in tqdm(range(test_steps)):
    # 프레임당 여러 번 움직여서 속도감 있게
    for _ in range(steps_per_frame):
        p_idx = np.random.randint(test_env.num_particles)
        r, c = test_env.particles[p_idx]
        state = test_env.get_observation(r, c)

        # 학습된 모델로 행동 결정 (Greedy)
        with torch.no_grad():
            action = policy_net(state).argmax().item()

        test_env.step(p_idx, action)

    frames.append(test_env.grid.copy())

# ==========================================
# 5. 애니메이션 생성
# ==========================================
cmap = colors.ListedColormap(['white', 'blue', 'black']) # 0:빈공간, 1:입자, 2:벽
bounds = [-0.5, 0.5, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(frames[0], cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Initial State")

def animate(idx):
    im.set_data(frames[idx])
    title.set_text(f"Trained Agent Simulation | Frame: {idx}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())

# 2. Kawasaki Dynamics

## 핵심 개념

지금까지의 "입자 + 빈 공간" 모델과 달리, 카와사키 동역학은 **두 종류의 스핀(±1)** 이 격자를 가득 채우고, 한 번의 업데이트는 **이웃한 두 셀의 값을 교환**하는 형태입니다.

### 정확한 정의

격자 위 Ising 해밀토니안

$$H(\sigma) = -J \sum_{\langle i,j \rangle} \sigma_i \sigma_j, \qquad \sigma_i \in \{+1, -1\}$$

위에서, **이웃한 두 사이트** $\langle i, j\rangle$ 를 무작위로 선택해 그 값을 교환합니다 ($\sigma_i \leftrightarrow \sigma_j$). 교환은 Metropolis 규칙에 따라 다음 확률로 받아들여집니다:

$$p_{\text{accept}} = \min\!\bigl(1,\, e^{-\beta \Delta E}\bigr), \qquad \beta = 1/T.$$

이 규칙의 두 가지 즉각적인 결과:

- 교환은 +1과 −1의 **개수를 보존**합니다 (Hohenberg–Halperin 분류의 Model B). 
- 같은 값을 가진 두 사이트의 교환은 항상 $\Delta E = 0$ 이며, 결과 격자는 변하지 않습니다 (no-op).

### 무한 온도 극한 ($\beta = 0$, 본 노트북에서 사용)

$\beta \to 0$ 일 때, $p_{\text{accept}} \to 1$ — 모든 교환을 항상 받아들입니다. 코드 관점에서:

> 무작위로 이웃 페어를 선택해 교환한다. 같은 값이면 결과는 변하지 않으므로 (no-op), 실질적으로 **다른 값의 페어만이 격자를 변화시킨다**.

따라서 아래 `step()` 메소드에서 `s_curr != s_neigh` 조건을 통해 같은 값의 케이스를 명시적으로 건너뛰는 것은, 결과 격자에 영향을 주지 않는 **연산 단축(short-circuit)** 입니다 (시간 매개변수화는 약간 달라집니다 — 표준 정의 대비 "한 step"이 무엇인지에 대한 정의가 다릅니다).

### 거시적 거동

이 노트북에서 시뮬레이션하는 $\beta = 0$ 의 경우, 거시적 농도장 $\rho(\vec{x}, t)$ 는 **선형 보존 확산방정식** 

$$\partial_t \rho = D\, \Delta \rho$$

으로 근사됩니다 (대칭 단순 배제 과정 SSEP의 hydrodynamic limit). 그러나 미시적으로는 *random walk가 아니라 stochastic exchange* 입니다.

> **유한 온도에서는** 위 단순 확산이 더 이상 성립하지 않습니다 — 일반적인 Kawasaki는 $\partial_t \rho = \nabla \cdot [M(\rho)\, \nabla \mu(\rho)]$ 형의 비선형 보존방정식 (Cahn-Hilliard 형) 으로 기술되며, $T < T_c$ 에서는 도메인 합쳐짐(coarsening, $\sim t^{1/3}$ growth)이 일어납니다. 이는 "추가 실험 아이디어 #3"의 영역입니다.

---

## 2-1. Ring 위 카와사키

### 2-1.1. 환경 정의

환형(구멍 뚫린 원판) 위에 +1(빨강)과 −1(파랑) 두 종이 섞여 있는 상태에서 시작합니다. 초기에는 −1이 왼쪽 영역에 모여 있습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from IPython.display import HTML
from tqdm import tqdm

class HighTempKawasaki:
    def __init__(self, N=60, ratio=0.8, wall_limit=20):
        self.N = N
        # 2: 벽, 1: 빈 공간, -1: 입자
        self.lattice = np.ones((N, N)) * 2

        # --- 1. 기하학적 구조 생성 (Punctured Disk / Annulus) ---
        cx, cy = N // 2, N // 2
        self.R_out = (N // 2) - 2
        self.R_in = self.R_out / 3.0 # 면적 1/9 조건에 맞게 설정

        self.valid_coords = [] # 입자가 존재할 수 있는 모든 좌표

        for x in range(N): # row
            for y in range(N): # col
                dist = np.sqrt((x - cx)**2 + (y - cy)**2)
                if self.R_in <= dist <= self.R_out:
                    self.lattice[x, y] = 1 # 빈 공간
                    self.valid_coords.append((x, y))
                else:
                    self.lattice[x, y] = 2 # 벽 (내부 구멍 및 외부 공간)

        # --- 2. 마이너스(-1, 파랑) 개수 계산 및 배치 ---
        num_total_valid = len(self.valid_coords)
        num_minus = int(num_total_valid * (1 - ratio))

        count = 0
        # 왼쪽 벽면 (x < wall_limit)에 -1 입자 집중 배치
        for col in range(wall_limit):
            for row in range(N):
                if self.lattice[row, col] == 1: # 유효한 빈 공간인 경우에만
                    if count < num_minus:
                        self.lattice[row, col] = -1
                        count += 1
                    else:
                        break
            if count >= num_minus:
                break
        # 남은 유효 공간은 +1 입자로 채움
        for x, y in self.valid_coords:
            if self.lattice[x, y] == 1: # 아직 비어있는 유효 공간
                self.lattice[x, y] = 1 # +1 입자 (빨강)로 채움

    def step(self):
        # 입자 (또는 +1 스핀)가 있는 유효한 좌표를 선택
        if not self.valid_coords: return # 유효한 공간이 없으면 스텝 진행 불가

        x, y = self.valid_coords[np.random.randint(len(self.valid_coords))]

        dx, dy = [(0, 1), (0, -1), (1, 0), (-1, 0)][np.random.randint(4)]

        nx, ny = x + dx, y + dy

        # 이동할 위치가 격자 범위 내에 있는지 확인
        if not (0 <= nx < self.N and 0 <= ny < self.N): return

        # 이동할 위치가 벽(2)이 아닌지 확인
        if self.lattice[nx, ny] == 2: return

        s_curr = self.lattice[x, y]
        s_neigh = self.lattice[nx, ny]

        # 서로 다른 스핀일 경우에만 교환
        if s_curr != s_neigh:
            self.lattice[x, y] = s_neigh
            self.lattice[nx, ny] = s_curr

### 2-1.2. 시뮬레이션 (Ring)

프레임마다 `N²` 번의 교환 시도를 수행합니다. 처음에는 왼쪽에 뚝쳐 있던 파란 영역이 점차 환형 전체에 퍼지며 두 색이 섞이는 과정이 보입니다.

In [ ]:

# ------------------------- 시뮬레이션 설정 ----------------------------
N = 60             # 격자 크기
FRAMES = 500       # 영상 프레임 수, 총 실험 시간과 비례
STEPS_PER_FRAME = N * N # 한 프레임당 전체 격자를 한 번 훑는 정도의 횟수
#-----------------------------------------------------------------------
sim = HighTempKawasaki(N=N, ratio=0.8, wall_limit=20)

# 색상 정의: -1(파랑), 1(빨강), 2(흰색 - 입자가 존재할 수 없는 공간)
cmap = colors.ListedColormap(['blue', 'red', 'white'])
bounds = [-1.5, 0.0, 1.5, 2.5]
norm = colors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(7, 7)) # Define fig and ax here

im = ax.imshow(sim.lattice, cmap=cmap, norm=norm)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("High T Diffusion (Random Walk) | Frame: 0")

print(f"고온 확산 시뮬레이션 진행 중... (총 {FRAMES} 프레임)")

def animate(frame):
    for _ in range(STEPS_PER_FRAME):
        sim.step()

    im.set_data(sim.lattice)
    title.set_text(f"Mixing Process | Frame: {frame}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=FRAMES, interval=50, blit=True)

plt.close()
HTML(anim.to_html5_video())

## 2-2. 토러스 위 카와사키

### 2-2.1. 환경 정의

이번에는 격자에 **주기적 경계조건(periodic boundary condition)** 을 줍니다. 위/아래·왼쪽/오른쪽이 모두 이어져 있는 토러스(torus, 도넛 표면) 위상입니다.

토러스에서는 "벽"이 없으므로, 끝까지 간 입자는 반대편으로 다시 들어옵니다. 두 클래스를 같은 이름(`HighTempKawasaki`)으로 새로 정의해 사용합니다 — 두 번째 정의가 활성 정의입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from tqdm import tqdm

class HighTempKawasaki:
    def __init__(self, N=60, ratio=0.8, wall_limit=20):
        self.N = N

        # 1. 초기화: 전체를 +1(빨강)로 채움
        self.lattice = np.ones((N, N))

        # 2. 마이너스(-1, 파랑) 개수 계산
        num_total = N * N
        num_minus = int(num_total * (1 - ratio))

        # 3. 왼쪽 벽면에 -1 배치 (초기 질서 상태)
        count = 0
        for col in range(wall_limit):
            for row in range(N):
                if count < num_minus:
                    self.lattice[row, col] = -1
                    count += 1
                else:
                    break
            if count >= num_minus:
                break

    def step(self):
        """
        고온(Infinite T) 카와사키 스텝:
        에너지 계산 없이, 서로 다른 스핀끼리 만나면 무조건 위치 교환 (확률 100%)
        """
        x, y = np.random.randint(0, self.N, 2)

        dx, dy = [(0, 1), (0, -1), (1, 0), (-1, 0)][np.random.randint(4)]

        new_x = x + dx
        new_y = y + dy

        nx = new_x % self.N
        ny = new_y % self.N

        s_curr = self.lattice[x, y]
        s_neigh = self.lattice[nx, ny]

        if s_curr != s_neigh:
            self.lattice[x, y] = s_neigh
            self.lattice[nx, ny] = s_curr


import numpy as np
import matplotlib.pyplot as plt

torus_sim = HighTempKawasaki(N=60, ratio=0.8, wall_limit=20)

plt.figure(figsize=(6, 6))
plt.imshow(torus_sim.lattice, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Initial State of HighTempKawasaki (Blue: -1 Blocks on Left Wall)")
plt.xlabel("Column")
plt.ylabel("Row")
plt.show()

In [ ]:
class HighTempKawasaki:
    def __init__(self, N=60, ratio=0.8, wall_limit=20):
        self.N = N

        self.lattice = np.ones((N, N))

        num_total = N * N
        num_minus = int(num_total * (1 - ratio))

        count = 0
        for col in range(wall_limit):
            for row in range(N):
                if count < num_minus:
                    self.lattice[row, col] = -1
                    count += 1
                else:
                    self.lattice[row, col] = 1
            if count >= num_minus:
                break

    def step(self):

        x, y = np.random.randint(0, self.N, 2)
        dx, dy = [(0, 1), (0, -1), (1, 0), (-1, 0)][np.random.randint(4)]
        nx, ny = (x + dx) % self.N, (y + dy) % self.N

        s_curr = self.lattice[x, y]
        s_neigh = self.lattice[nx, ny]

        if s_curr != s_neigh:
            self.lattice[x, y] = s_neigh
            self.lattice[nx, ny] = s_curr


### 2-2.2. 시뮬레이션 — 평면 표시

먼저 격자를 평면 그대로 보면서 섞임 과정을 관찰합니다.

In [ ]:

# ------------------------- 시뮬레이션 설정 ----------------------------
N = 60             # 격자 크기
FRAMES = 500       # 영상 프레임 수, 총 실험 시간과 비례
STEPS_PER_FRAME = N * N # 한 프레임당 전체 격자를 한 번 훑는 정도의 횟수
#-----------------------------------------------------------------------
torus_sim = HighTempKawasaki(N=N, ratio=0.8, wall_limit=20)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(torus_sim.lattice, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("High T Diffusion (Random Walk) | Frame: 0")

print(f"고온 확산 시뮬레이션 진행 중... (총 {FRAMES} 프레임)")

def animate(frame):
    for _ in range(STEPS_PER_FRAME):
        torus_sim.step()

    im.set_data(torus_sim.lattice)
    title.set_text(f"Mixing Process | Frame: {frame}")
    return im, title

anim = animation.FuncAnimation(fig, animate, frames=FRAMES, interval=50, blit=True)

plt.close()
HTML(anim.to_html5_video())

### 2-2.3. 3D 시각화 (정면 뷰)

격자를 토러스 표면에 매핑해 3D로 보여 줍니다.

$$ X = (R + r\cos v)\cos u, \quad Y = (R + r\cos v)\sin u, \quad Z = r\sin v $$

여기서 $R$ 은 큰 반경, $r$ 은 작은 반경, $u, v \in [0, 2\pi)$ 입니다. 격자의 두 좌표가 각각 $u, v$ 에 대응되어, 격자의 "양 끝이 이어진 토폴로지"가 자연스럽게 표현됩니다.

> **시각화 주의** — 시뮬레이션은 *평면 토러스* (메트릭이 균일한 $\mathbb{R}^2 / N\mathbb{Z}^2$) 위에서 진행되지만, 위 임베딩은 $\mathbb{R}^3$ 에 휘어 넣은 것이라 곡률이 균일하지 않습니다 (외측이 내측보다 면적이 큼). 그 결과 같은 격자 밀도라도 외측에서는 시각적으로 더 성기게, 내측에서는 빽빽하게 보일 수 있습니다 — 이는 *순전한 시각화 왜곡* 이며, 시뮬레이션 자체는 평면 위에서 동등하게 진행됩니다.

In [ ]:
#----------------- 시뮬레이션 설정 -----------------------------------
N = 60             # 격자 크기
ANIMATION_FRAMES = 1000 # 영상 프레임 수
STEPS_PER_FRAME = N * N // 2 # 한 프레임당 시뮬레이션 스텝 수 (N*N은 모든 격자점을 한 번씩 시도하는 수준)
#---------------------------------------------------------------------
# HighTempKawasaki 시뮬레이션 인스턴스 생성
torus_sim = HighTempKawasaki(N=N, ratio=0.8, wall_limit=20)

# --- 3D 토러스 고정 좌표 계산 (한 번만 계산) ---
R = 1.0  # 주 반지름 (Major radius)
r = 0.4  # 부 반지름 (Minor radius)
i, j = np.meshgrid(np.arange(N), np.arange(N))
u = 2 * np.pi * i / N
v = 2 * np.pi * j / N
X = (R + r * np.cos(v)) * np.cos(u)
Y = (R + r * np.cos(v)) * np.sin(u)
Z = r * np.sin(v)
X_flat = X.flatten()
Y_flat = Y.flatten()
Z_flat = Z.flatten()

# --- 애니메이션 프레임 데이터 수집 ---
all_torus_frames = []

print(f"토러스 위 입자 움직임 데이터 수집 중... (총 {ANIMATION_FRAMES} 프레임)")
for _ in tqdm(range(ANIMATION_FRAMES)):
    for _ in range(STEPS_PER_FRAME):
        torus_sim.step() # 시뮬레이션 스텝 진행
    all_torus_frames.append(torus_sim.lattice.flatten()) # 현재 격자 상태 저장

# --- 애니메이션 생성 ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# 뷰 각도 설정 (elev=45도, azim=45도)
ax.view_init(elev=60, azim=45)

cmap = colors.ListedColormap(['blue', 'yellow'])
norm = colors.BoundaryNorm([-1.5, 0.0, 1.5], cmap.N)

# 초기 산점도
scatter = ax.scatter(X_flat, Y_flat, Z_flat, c=all_torus_frames[0], cmap=cmap, norm=norm, s=10, alpha=0.8)

ax.set_title(f'Kawasaki Simulation on Torus | Frame: 0')
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.set_axis_off()

def animate(frame_idx):
    scatter.set_array(all_torus_frames[frame_idx])
    ax.set_title(f'Kawasaki Simulation on Torus | Frame: {frame_idx}')
    return scatter,

print("토러스 애니메이션 렌더링 중...")
anim = animation.FuncAnimation(fig, animate, frames=len(all_torus_frames), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())


### 2-2.4. 3D 시각화 (대각선 뷰)

동일한 데이터를 X축 30°, Z축 45° 로 회전한 시점에서 다시 그립니다. 같은 시뮬레이션도 시점에 따라 매우 다르게 느껴진다는 점을 확인할 수 있습니다.

In [ ]:
#------------------------- 시뮬레이션 설정 -----------------------
N = 60             # 격자 크기
ANIMATION_FRAMES = 500 # 영상 프레임 수
STEPS_PER_FRAME = N * N // 2 # 한 프레임당 시뮬레이션 스텝 수 (N*N은 모든 격자점을 한 번씩 시도하는 수준)
#-----------------------------------------------------------------

# HighTempKawasaki 시뮬레이션 인스턴스 생성
torus_sim = HighTempKawasaki(N=N, ratio=0.8, wall_limit=20)

# --- 3D 토러스 고정 좌표 계산 (한 번만 계산) ---
R = 1.0  # 주 반지름 (Major radius)
r = 0.4  # 부 반지름 (Minor radius)
i, j = np.meshgrid(np.arange(N), np.arange(N))
u = 2 * np.pi * i / N
v = 2 * np.pi * j / N
X_orig = (R + r * np.cos(v)) * np.cos(u)
Y_orig = (R + r * np.cos(v)) * np.sin(u)
Z_orig = r * np.sin(v)

# 대각선 뷰를 위한 객체 자체의 회전 적용 (좌표 변환)
# 예시: X축을 중심으로 30도, Z축을 중심으로 45도 회전
angle_x_rot = np.deg2rad(30) # X축 기준 회전 각도
angle_z_rot = np.deg2rad(45) # Z축 기준 회전 각도

# Rotation around X-axis
X_temp = X_orig
Y_temp = Y_orig * np.cos(angle_x_rot) - Z_orig * np.sin(angle_x_rot)
Z_temp = Y_orig * np.sin(angle_x_rot) + Z_orig * np.cos(angle_x_rot)

# Rotation around Z-axis
X_rotated = X_temp * np.cos(angle_z_rot) - Y_temp * np.sin(angle_z_rot)
Y_rotated = X_temp * np.sin(angle_z_rot) + Y_temp * np.cos(angle_z_rot)
Z_rotated = Z_temp

X_flat = X_rotated.flatten()
Y_flat = Y_rotated.flatten()
Z_flat = Z_rotated.flatten()

# --- 애니메이션 프레임 데이터 수집 ---
all_torus_frames = []

print(f"토러스 위 입자 움직임 데이터 수집 중... (총 {ANIMATION_FRAMES} 프레임)")
for _ in tqdm(range(ANIMATION_FRAMES)):
    for _ in range(STEPS_PER_FRAME):
        torus_sim.step() # 시뮬레이션 스텝 진행
    all_torus_frames.append(torus_sim.lattice.flatten()) # 현재 격자 상태 저장

# --- 애니메이션 생성 ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# 뷰 각도 설정 (elev=60도, azim=45도) - 객체 회전 후의 시점
ax.view_init(elev=60, azim=45)

# 색상 정의: -1(파랑), 1(노랑, 시각화 편의성), 2(흰색 - 입자가 존재할 수 없는 공간)
cmap = colors.ListedColormap(['blue', 'yellow', 'white'])
norm = colors.BoundaryNorm([-1.5, 0.0, 1.5, 2.5], cmap.N)

# 초기 산점도
scatter = ax.scatter(X_flat, Y_flat, Z_flat, c=all_torus_frames[0], cmap=cmap, norm=norm, s=10, alpha=0.8)

ax.set_title(f'Kawasaki Simulation on Rotated Torus | Frame: 0')
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.set_axis_off()

def animate(frame_idx):
    scatter.set_array(all_torus_frames[frame_idx])
    ax.set_title(f'Kawasaki Simulation on Rotated Torus | Frame: {frame_idx}')
    return scatter,

print("토러스 애니메이션 렌더링 중...")
anim = animation.FuncAnimation(fig, animate, frames=len(all_torus_frames), interval=100, blit=True)
plt.close()
HTML(anim.to_html5_video())

## 정리

- **Random Walk** 만으로는 시스템이 균일하게 "섞이는" 데까지 도달합니다 — 이상적인 분포는 균일.
- 같은 무작위 시도에 "평가 함수"를 붙이면(=Greedy) 시스템은 균일이 아닌 **분리·정렬** 같은 비자명한 상태로 수렴합니다.
- DQN은 사람이 명시한 그리디 규칙 없이도, 경험에서 비슷한 분리 정책을 학습할 수 있음을 보였습니다.
- Kawasaki 동역학은 "입자 보존"이라는 제약 하의 stochastic exchange이며, 위상(공간 모양)에 따라 섞이는 양상이 달라집니다. 미시적으로는 random walk가 아닌 교환 동역학이지만, 거시적 농도장 수준에서는 보존 확산방정식으로 환원됩니다.

## 추가 실험 아이디어

1. `particle_ratio` 를 0.05, 0.5, 0.7 로 바꿔 보고 Greedy 단계의 결과를 비교하세요.
2. 1-4의 DQN 보상 함수를 "이웃과 가까울수록 좋다" 로 뒤집어 보세요. 학습된 정책이 어떻게 달라집니까?
3. 2-1 카와사키에서 유한 온도의 Metropolis 규칙을 도입해 보세요: 이웃 페어를 무작위로 선택하고, 그 교환에 따른 에너지 변화 $\Delta E$ 를 계산한 뒤, 확률 $\min(1, e^{-\beta \Delta E})$ 로 받아들이는 것입니다 ($\beta = 1/T$). 충분히 낮은 $T$ 에서는 도메인이 형성되며, 점점 큰 도메인으로 합쳐지는 (coarsening) 거동이 관찰될 것입니다.